# Day 3: PyTorch Basics and Re-implementation (Notebook 2)

Aaj ka goal: PyTorch ke basics (tensors, autograd, nn.Module, DataLoader), Day 2 ka network PyTorch me banana, NumPy se compare karna, aur SGD vs Adam plot karna.

In [4]:
import copy
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split

torch.manual_seed(42)
np.random.seed(42)
print("torch version:", torch.__version__)

torch version: 2.15.0+cpu


## 1. Tensors

Tensor = NumPy array jaisa, lekin GPU par chal sakta hai aur gradient track kar sakta hai.

In [5]:
a = torch.tensor([[1., 2.], [3., 4.]])
b = torch.ones(2, 2)

print("a:", a, sep="\n")
print("shape:", a.shape, " dtype:", a.dtype)
print("a + b:", a + b, sep="\n")
print("a @ b (matrix multiply):", a @ b, sep="\n")

# NumPy <-> Tensor
arr = a.numpy()
back = torch.from_numpy(arr)
print(type(arr), type(back))

a:
tensor([[1., 2.],
        [3., 4.]])
shape: torch.Size([2, 2])  dtype: torch.float32
a + b:
tensor([[2., 3.],
        [4., 5.]])
a @ b (matrix multiply):
tensor([[3., 3.],
        [7., 7.]])
<class 'numpy.ndarray'> <class 'torch.Tensor'>


## 2. Autograd

PyTorch computation graph banata hai, aur `.backward()` chalane par khud gradient nikalta hai.
Day 2 me jo backward tumne haath se likha, ye wahi kaam automatic karta hai.

In [6]:
w = torch.tensor(2.0, requires_grad=True)
x = torch.tensor(3.0)

y = w * x + 1          # y = 7
loss = (y - 10) ** 2   # loss = 9
loss.backward()

# dL/dw = 2 * (y - 10) * x = 2 * (-3) * 3 = -18
print("w.grad:", w.grad.item())

w.grad: -18.0


## 3. Data (same as Day 2) aur DataLoader

`Dataset` = data ka collection, `DataLoader` = usay batches me tod kar (aur shuffle karke) deta hai.

In [7]:
X, y = make_moons(n_samples=500, noise=0.2, random_state=42)
y = y.reshape(-1, 1)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

X_train_t = torch.tensor(X_train, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.float32)
X_test_t  = torch.tensor(X_test,  dtype=torch.float32)
y_test_t  = torch.tensor(y_test,  dtype=torch.float32)

train_ds = TensorDataset(X_train_t, y_train_t)
batch_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
full_loader  = DataLoader(train_ds, batch_size=len(train_ds), shuffle=False)

xb, yb = next(iter(batch_loader))
print("one batch:", xb.shape, yb.shape)

one batch: torch.Size([32, 2]) torch.Size([32, 1])


## 4. Model: nn.Module

`nn.Linear(n_in, n_out)` = `X @ W + b` (jo tumne Day 1 me haath se likha).

Output layer par sigmoid **nahi** lagate, kyunke `BCEWithLogitsLoss` andar sigmoid + BCE ek saath karta hai (numerically zyada stable). Model raw scores (logits) deta hai.

In [8]:
class Net(nn.Module):
    def __init__(self, n_in=2, n_hidden=16):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_in, n_hidden),
            nn.ReLU(),
            nn.Linear(n_hidden, 1),
        )

    def forward(self, x):
        return self.net(x)   # logits

model = Net()
print(model)
print("parameters:", sum(p.numel() for p in model.parameters()))

Net(
  (net): Sequential(
    (0): Linear(in_features=2, out_features=16, bias=True)
    (1): ReLU()
    (2): Linear(in_features=16, out_features=1, bias=True)
  )
)
parameters: 65


## 5. Standard training loop

Har step par yehi 5 cheezein: **forward -> loss -> zero_grad -> backward -> step**

In [ ]:
def train_loop(model, optimizer, loader, epochs):
    criterion = nn.BCEWithLogitsLoss()
    history = []
    for epoch in range(epochs):
        model.train()
        total = 0.0
        for xb, yb in loader:
            logits = model(xb)               # 1. forward
            loss = criterion(logits, yb)     # 2. loss
            optimizer.zero_grad()            # 3. purane gradients saaf
            loss.backward()                  # 4. backward (autograd)
            optimizer.step()                 # 5. weights update
            total += loss.item() * len(xb)
        history.append(total / len(loader.dataset))
    return history

def accuracy(model, X, y):
    model.eval()
    with torch.no_grad():
        preds = (torch.sigmoid(model(X)) > 0.5).float()
    return (preds == y).float().mean().item()

# Mini-batch training with Adam
torch.manual_seed(0)
model = Net()
opt = torch.optim.Adam(model.parameters(), lr=0.01)
history = train_loop(model, opt, batch_loader, epochs=100)

print(f"train acc: {accuracy(model, X_train_t, y_train_t):.3f}")
print(f"test  acc: {accuracy(model, X_test_t, y_test_t):.3f}")

plt.figure(figsize=(6, 3.5))
plt.plot(history)
plt.xlabel("epoch"); plt.ylabel("loss"); plt.title("PyTorch (mini-batch, Adam)")
plt.grid(alpha=0.3)
plt.show()

In [ ]:
def plot_boundary_torch(model, X, y, title):
    xx, yy = np.meshgrid(np.linspace(X[:, 0].min() - 0.5, X[:, 0].max() + 0.5, 200),
                         np.linspace(X[:, 1].min() - 0.5, X[:, 1].max() + 0.5, 200))
    grid = torch.tensor(np.c_[xx.ravel(), yy.ravel()], dtype=torch.float32)
    model.eval()
    with torch.no_grad():
        zz = torch.sigmoid(model(grid)).numpy().reshape(xx.shape)
    plt.figure(figsize=(6, 4))
    plt.contourf(xx, yy, zz, levels=20, cmap="RdBu", alpha=0.6)
    plt.scatter(X[:, 0], X[:, 1], c=y.ravel(), cmap="RdBu", edgecolor="k", s=20)
    plt.title(title)
    plt.show()

plot_boundary_torch(model, X_test, y_test, "PyTorch model (test data)")

## 6. NumPy vs PyTorch comparison

Fair comparison ke liye dono me same setup: full-batch, plain SGD, lr=0.5, 2000 epochs.
Pehle Day 2 ka NumPy MLP (copy), phir PyTorch version.

In [ ]:
def sigmoid_np(z):
    return 1 / (1 + np.exp(-z))

def relu_np(z):
    return np.maximum(0, z)

class MLP:
    def __init__(self, n_in, n_hidden, n_out, lr=0.1, seed=0):
        rng = np.random.default_rng(seed)
        self.lr = lr
        self.W1 = rng.standard_normal((n_in, n_hidden)) * np.sqrt(2 / n_in)
        self.b1 = np.zeros((1, n_hidden))
        self.W2 = rng.standard_normal((n_hidden, n_out)) * np.sqrt(2 / n_hidden)
        self.b2 = np.zeros((1, n_out))

    def forward(self, X):
        self.X = X
        self.Z1 = X @ self.W1 + self.b1
        self.A1 = relu_np(self.Z1)
        self.Z2 = self.A1 @ self.W2 + self.b2
        self.A2 = sigmoid_np(self.Z2)
        return self.A2

    def loss(self, y_hat, y, eps=1e-12):
        y_hat = np.clip(y_hat, eps, 1 - eps)
        return -np.mean(y * np.log(y_hat) + (1 - y) * np.log(1 - y_hat))

    def backward(self, y):
        m = y.shape[0]
        dZ2 = (self.A2 - y) / m
        self.dW2 = self.A1.T @ dZ2
        self.db2 = dZ2.sum(axis=0, keepdims=True)
        dZ1 = (dZ2 @ self.W2.T) * (self.Z1 > 0)
        self.dW1 = self.X.T @ dZ1
        self.db1 = dZ1.sum(axis=0, keepdims=True)

    def update(self):
        self.W1 -= self.lr * self.dW1
        self.b1 -= self.lr * self.db1
        self.W2 -= self.lr * self.dW2
        self.b2 -= self.lr * self.db2

    def predict(self, X):
        return (self.forward(X) > 0.5).astype(int)

# --- NumPy ---
np_model = MLP(2, 16, 1, lr=0.5)
np_losses = []
for epoch in range(2000):
    y_hat = np_model.forward(X_train)
    np_losses.append(np_model.loss(y_hat, y_train))
    np_model.backward(y_train)
    np_model.update()
np_test_acc = (np_model.predict(X_test) == y_test).mean()

# --- PyTorch ---
torch.manual_seed(0)
pt_model = Net()
pt_opt = torch.optim.SGD(pt_model.parameters(), lr=0.5)
pt_losses = train_loop(pt_model, pt_opt, full_loader, epochs=2000)
pt_test_acc = accuracy(pt_model, X_test_t, y_test_t)

print(f"NumPy   final loss: {np_losses[-1]:.4f}   test acc: {np_test_acc:.3f}")
print(f"PyTorch final loss: {pt_losses[-1]:.4f}   test acc: {pt_test_acc:.3f}")

plt.figure(figsize=(7, 4))
plt.plot(np_losses, label="NumPy (from scratch)")
plt.plot(pt_losses, "--", label="PyTorch")
plt.xlabel("epoch"); plt.ylabel("loss"); plt.title("NumPy vs PyTorch")
plt.legend(); plt.grid(alpha=0.3)
plt.show()

Dono ke curves aur accuracy lagbhag same aane chahiye. Chhota farq is liye hai ke initial weights alag hain. Isse pata chalta hai ke tumhara from-scratch backprop sahi hai.

## 7. SGD vs Adam

- **SGD:** `w = w - lr * grad`
- **SGD + momentum:** pichle gradients ka average bhi use karta hai, isliye smooth aur tez
- **Adam:** momentum + har weight ke liye apna adaptive learning rate

Teeno ko **same initial weights** se shuru karte hain taake comparison fair ho.

In [ ]:
torch.manual_seed(1)
init_model = Net()

optimizers = {
    "SGD (lr=0.1)":            lambda p: torch.optim.SGD(p, lr=0.1),
    "SGD + momentum (0.9)":    lambda p: torch.optim.SGD(p, lr=0.1, momentum=0.9),
    "Adam (lr=0.01)":          lambda p: torch.optim.Adam(p, lr=0.01),
}

results = {}
for name, make_opt in optimizers.items():
    m = copy.deepcopy(init_model)
    opt = make_opt(m.parameters())
    hist = train_loop(m, opt, full_loader, epochs=300)
    acc = accuracy(m, X_test_t, y_test_t)
    results[name] = hist
    print(f"{name:24s} final loss: {hist[-1]:.4f}   test acc: {acc:.3f}")

plt.figure(figsize=(7, 4))
for name, hist in results.items():
    plt.plot(hist, label=name)
plt.xlabel("epoch"); plt.ylabel("loss"); plt.title("SGD vs Adam")
plt.legend(); plt.grid(alpha=0.3)
plt.show()

## 8. Experiments (khud karo)

1. Adam ka `lr` 0.001, 0.01, 0.1 karke dekho.
2. SGD ka `lr` 0.01 aur 1.0 karo. Slow ya unstable kaun hai?
3. `batch_size` 8, 32, 400 karo (mini-batch vs full-batch), loss curve kaise badalti hai?
4. Training loop me `optimizer.zero_grad()` hata kar dekho, kya hota hai? (gradients jama hote jate hain)
5. `n_hidden` 2 aur 64 karo.

## 9. Notes (apne words me likho)

- Tensor aur autograd kya hain:
- `nn.Module` aur `nn.Linear`:
- Training loop ke 5 steps aur har ek ka kaam:
- Adam SGD se aksar tez kyun hota hai: